# Synthetic EHR Dataset Generator

Generates a synthetic dataset of Electronic Health Records (EHRs) from the
educational disease–symptom subset, with symptom frequencies **proportional to importance**.

---

## Output schema

| Column | Description |
|--------|-------------|
| `patient_id` | Unique patient code (e.g. `P0001`) |
| `ehr_id` | Unique EHR code (e.g. `EHR00001`). One EHR = one disease episode. |
| `disease` | The disease recorded in this EHR |
| `symptom` | One symptom reported in this EHR (one row per symptom) |

Each patient may have multiple EHRs (different episodes). Each EHR has exactly
one disease and one or more symptom rows.

---

## Sampling model

### Why not use importance directly as a Bernoulli probability?

The `importance` values in the source file are low (0.008 – 0.45) and each disease
has only 2–6 symptoms in the educational subset. This means the probability of
drawing **zero** symptoms for an EHR is very high (up to 94% for heart attack).
Naively rejecting empty draws would condition the proportions upward and destroy
the target distribution.

### Solution: weighted count-first sampling

For each EHR of disease *d*:

1. Draw how many symptoms the EHR will report, between configurable minimum
   and maximum values (capped by the number of known symptoms for that disease).
2. Draw that many unique symptoms without replacement, sampled proportionally
   to the disease-specific importance weights.

This produces a wider range of symptom counts per EHR while keeping symptoms
with higher importance more likely to appear. Because symptoms are unique within
an EHR and some diseases have only a few known symptoms, the recovered aggregate
proportions are approximate rather than an exact Bernoulli identity.

### What 'proportional to importance' means in the output

For a given disease, if you count how many EHRs report each symptom and normalise
by the total symptom occurrences, the resulting proportions will match
`importance_s / Σ importance` — i.e. more important symptoms appear in a larger
share of records, with the same relative ratios as in the source data.

In [17]:
import pandas as pd
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_FILE       = 'educational_subset.csv'   # output of the previous notebook
OUTPUT_FILE      = 'synthetic_ehr_evaluate.csv'

MIN_EHRS_PER_DISEASE = 40  # lower bound for EHRs generated per disease
MAX_EHRS_PER_DISEASE = 60  # upper bound for EHRs generated per disease
N_PATIENTS        = 80   # total unique patients (some will have >1 EHR)
MIN_SYMPTOMS_PER_EHR = 1  # lower bound for symptoms reported in one EHR
MAX_SYMPTOMS_PER_EHR = 4  # upper bound, capped by symptoms available per disease
RANDOM_SEED       = 607

rng = np.random.default_rng(RANDOM_SEED)

In [18]:
# ── 1. Load the educational subset ────────────────────────────────────────────
df_ref = pd.read_csv(INPUT_FILE)
print(f'Loaded {len(df_ref)} (disease, symptom) pairs.')
print(f'Diseases : {df_ref["disease"].nunique()}')
print(f'Symptoms : {df_ref["symptom"].nunique()}')
df_ref.head()

Loaded 31 (disease, symptom) pairs.
Diseases : 10
Symptoms : 10


,disease,symptom,importance
0,pneumonia,fever,0.208
1,pneumonia,coughing,0.402
2,pneumonia,shortness of breath,0.156
3,migraine,headache,0.384
4,migraine,seizures,0.017


In [19]:
# ── 2. Pre-compute per-disease symptom tables ─────────────────────────────────
# For each disease, store an ordered array of symptoms and their importances.
disease_profiles = {}
for disease, group in df_ref.groupby('disease'):
    syms  = group['symptom'].tolist()
    imps  = group['importance'].to_numpy(dtype=float)
    weights = imps / imps.sum()   # normalised importance weights
    p_empty = float(np.prod(1.0 - imps))  # P(all Bernoulli draws = 0)
    disease_profiles[disease] = dict(
        symptoms=syms,
        importances=imps,
        weights=weights,
        p_empty=p_empty,
    )

print('Disease profiles (p_empty = probability that a raw Bernoulli draw gives zero symptoms):')
for d, prof in disease_profiles.items():
    print(f'  {d:<30}  n_symptoms={len(prof["symptoms"])}  '
          f'sum_importance={prof["importances"].sum():.3f}  '
          f'p_empty={prof["p_empty"]:.3f}')

Disease profiles (p_empty = probability that a raw Bernoulli draw gives zero symptoms):
  anaphylaxis                     n_symptoms=3  sum_importance=0.273  p_empty=0.747
  appendicitis                    n_symptoms=2  sum_importance=0.118  p_empty=0.884
  gout                            n_symptoms=2  sum_importance=0.257  p_empty=0.746
  heart attack                    n_symptoms=2  sum_importance=0.061  p_empty=0.940
  major depression                n_symptoms=4  sum_importance=0.459  p_empty=0.559
  meningitis                      n_symptoms=6  sum_importance=1.155  p_empty=0.246
  migraine                        n_symptoms=5  sum_importance=0.663  p_empty=0.452
  pneumonia                       n_symptoms=3  sum_importance=0.766  p_empty=0.400
  stroke                          n_symptoms=2  sum_importance=0.120  p_empty=0.883
  urinary tract infection         n_symptoms=2  sum_importance=0.217  p_empty=0.793


In [20]:
# ── 3. Sampling function ──────────────────────────────────────────────────────

def sample_symptoms(disease: str, rng: np.random.Generator) -> list[str]:
    """
    Sample the set of symptoms reported in one EHR for *disease*.

    Algorithm
    ---------
    1. Draw how many symptoms this EHR will report.
    2. Sample that many unique symptoms without replacement, using
       importance-normalised weights.

    Returns
    -------
    list[str]  — symptom names (no duplicates, length between configured bounds).
    """
    prof    = disease_profiles[disease]
    syms    = prof['symptoms']
    weights = prof['weights']

    min_count = max(1, min(MIN_SYMPTOMS_PER_EHR, len(syms)))
    max_count = max(min_count, min(MAX_SYMPTOMS_PER_EHR, len(syms)))
    symptom_count = rng.integers(min_count, max_count + 1)

    sampled_idx = rng.choice(len(syms), size=symptom_count, replace=False, p=weights)

    return [syms[i] for i in sampled_idx]

In [21]:
# ── 4. Generate EHR records ───────────────────────────────────────────────────
diseases = sorted(disease_profiles.keys())
ehrs_per_disease = {
    disease: rng.integers(MIN_EHRS_PER_DISEASE, MAX_EHRS_PER_DISEASE + 1)
    for disease in diseases
}
total_ehrs = sum(ehrs_per_disease.values())

# Assign patient IDs: shuffle EHR indices and map to N_PATIENTS patients
# so that patients are distributed roughly evenly and some have multiple EHRs.
ehr_indices   = np.arange(total_ehrs)
patient_ids_raw = (rng.permutation(total_ehrs) % N_PATIENTS)  # 0-based

records = []
ehr_counter = 0

for disease in diseases:
    for _ in range(ehrs_per_disease[disease]):
        patient_id = f'P1{patient_ids_raw[ehr_counter]:03d}'
        ehr_id     = f'EHR1{ehr_counter:04d}'
        symptoms   = sample_symptoms(disease, rng)

        for symptom in symptoms:
            records.append({
                'patient_id': patient_id,
                'ehr_id'    : ehr_id,
                'disease'   : disease,
                'symptom'   : symptom,
            })
        ehr_counter += 1

df_out = pd.DataFrame(records, columns=['patient_id', 'ehr_id', 'disease', 'symptom'])
print(f'Generated {ehr_counter:,} EHRs  →  {len(df_out):,} rows  ({len(df_out)/ehr_counter:.2f} symptoms/EHR on average)')
df_out.head(12)

Generated 527 EHRs  →  1,084 rows  (2.06 symptoms/EHR on average)


,patient_id,ehr_id,disease,symptom
0,P1018,EHR10000,anaphylaxis,swelling
1,P1018,EHR10000,anaphylaxis,shortness of breath
2,P1018,EHR10000,anaphylaxis,seizures
3,P1038,EHR10001,anaphylaxis,swelling
4,P1038,EHR10001,anaphylaxis,shortness of breath
5,P1038,EHR10001,anaphylaxis,seizures
6,P1054,EHR10002,anaphylaxis,swelling
7,P1054,EHR10002,anaphylaxis,shortness of breath
8,P1054,EHR10002,anaphylaxis,seizures
9,P1033,EHR10003,anaphylaxis,shortness of breath


In [22]:
# ── 5. Validation: recovered proportions vs target importances ────────────────
print('Symptom frequency proportions (recovered vs target importance weights)\n')
print(f'{"Disease":<30} {"Symptom":<28} {"Recovered":>10} {"Target":>10} {"Error":>8}')
print('-' * 90)

max_errors = {}
for disease, prof in disease_profiles.items():
    syms    = prof['symptoms']
    weights = prof['weights']   # target relative proportions

    sub = df_out[df_out['disease'] == disease]
    total_occurrences = len(sub)

    errs = []
    for sym, w_target in zip(syms, weights):
        count     = (sub['symptom'] == sym).sum()
        recovered = count / total_occurrences
        err       = abs(recovered - w_target)
        errs.append(err)
        print(f'{disease:<30} {sym:<28} {recovered:>10.4f} {w_target:>10.4f} {err:>8.4f}')

    max_errors[disease] = max(errs)

print()
print('Maximum relative-proportion error per disease:')
for d, e in sorted(max_errors.items(), key=lambda x: -x[1]):
    print(f'  {d:<30} {e:.4f}')
print(f'\nOverall max error: {max(max_errors.values()):.4f}')

Symptom frequency proportions (recovered vs target importance weights)

Disease                        Symptom                       Recovered     Target    Error
------------------------------------------------------------------------------------------
anaphylaxis                    seizures                         0.2422     0.0733   0.1689
anaphylaxis                    shortness of breath              0.3594     0.3626   0.0033
anaphylaxis                    swelling                         0.3984     0.5641   0.1657
appendicitis                   fever                            0.5904     0.8136   0.2232
appendicitis                   pain during urination            0.4096     0.1864   0.2232
gout                           fever                            0.3718     0.0428   0.3290
gout                           swelling                         0.6282     0.9572   0.3290
heart attack                   depression                       0.4405     0.3279   0.1126
heart attack      

In [23]:
# ── 6. Summary statistics ─────────────────────────────────────────────────────
n_patients = df_out['patient_id'].nunique()
n_ehrs     = df_out['ehr_id'].nunique()
ehrs_per_patient = df_out.groupby('patient_id')['ehr_id'].nunique()
syms_per_ehr     = df_out.groupby('ehr_id')['symptom'].count()

print('=== Dataset summary ===')
print(f'  Unique patients       : {n_patients}')
print(f'  Unique EHRs           : {n_ehrs}')
print(f'  Total rows            : {len(df_out):,}')
print(f'  EHRs per patient      : min={ehrs_per_patient.min()}  '
      f'mean={ehrs_per_patient.mean():.2f}  max={ehrs_per_patient.max()}')
print(f'  Symptoms per EHR      : min={syms_per_ehr.min()}  '
      f'mean={syms_per_ehr.mean():.2f}  max={syms_per_ehr.max()}')
print()
print('EHRs per disease:')
print(df_out.drop_duplicates('ehr_id')['disease'].value_counts().to_string())

=== Dataset summary ===
  Unique patients       : 80
  Unique EHRs           : 527
  Total rows            : 1,084
  EHRs per patient      : min=6  mean=6.59  max=7
  Symptoms per EHR      : min=1  mean=2.06  max=4

EHRs per disease:
disease
anaphylaxis                60
meningitis                 60
migraine                   59
gout                       54
stroke                     54
appendicitis               53
heart attack               53
major depression           49
urinary tract infection    45
pneumonia                  40


In [24]:
# ── 7. Save to CSV ────────────────────────────────────────────────────────────
df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  ({len(df_out):,} rows)')

Saved -> synthetic_ehr_evaluate.csv  (1,084 rows)
